In [ ]:
!pip install -qU langchain-google-genai langgraph yfinance duckduckgo-search mlflow pydantic

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field

class MarketResearchOutput(BaseModel):
    company_name: str = Field(description="Name of the company")
    stock_code: str = Field(description="Stock ticker symbol")
    current_price: float = Field(description="Current stock price")
    price_change_7d: str = Field(description="Price change percentage last 7 days")
    price_change_30d: str = Field(description="Price change percentage last 30 days")
    fifty_two_week_high: float = Field(description="52-week high price")
    fifty_two_week_low: float = Field(description="52-week low price")
    news_summary: str = Field(description="Summary of recent news headlines")
    sentiment: str = Field(description="Overall sentiment: Positive, Negative, or Neutral")
    confidence_score: float = Field(description="Confidence score between 0.0 and 1.0")
    people_names: List[str] = Field(description="Names of key people mentioned")
    places_names: List[str] = Field(description="Names of places/locations mentioned")
    other_companies_referred: List[str] = Field(description="Other companies mentioned in the news")
    related_industries: List[str] = Field(description="Industries related to the company/news")
    market_implications: str = Field(description="Market implications of the current news and trends")
    investment_recommendation: str = Field(description="Investment recommendation: Buy, Hold, or Avoid")
    recommendation_rationale: str = Field(description="Rationale behind the recommendation")

In [ ]:
import yfinance as yf
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash") # As per assignment spec

def resolve_ticker(company_name: str):
    # Prompting Gemini to get the ticker
    response = llm.invoke(f"Return only the stock ticker symbol for {company_name}. No prose.")
    return response.content.strip().upper()

def fetch_news(company_name: str):
    search = DuckDuckGoSearchResults()
    results = search.run(f"recent financial news for {company_name}")
    return results

In [ ]:
def fetch_stock_price(ticker: str):
    stock = yf.Ticker(ticker)
    hist_7d = stock.history(period="7d")
    hist_30d = stock.history(period="30d")
    info = stock.info

    # Calculate percentage changes
    change_7d = ((hist_7d['Close'].iloc[-1] - hist_7d['Close'].iloc[0]) / hist_7d['Close'].iloc[0]) * 100
    change_30d = ((hist_30d['Close'].iloc[-1] - hist_30d['Close'].iloc[0]) / hist_30d['Close'].iloc[0]) * 100

    return {
        "current_price": info.get('currentPrice', hist_7d['Close'].iloc[-1]),
        "price_change_7d": f"{change_7d:+.2f}%",
        "price_change_30d": f"{change_30d:+.2f}%",
        "high_52": info.get('fiftyTwoWeekHigh'),
        "low_52": info.get('fiftyTwoWeekLow')
    }

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
import mlflow

# MLflow setup for Step 6[cite: 1]
mlflow.set_experiment("Market_Sentiment_Analyzer")
mlflow.langchain.autolog()

class AgentState(TypedDict):
    company_input: str
    ticker: str
    news_data: str
    stock_data: dict
    final_report: dict

def node_resolve_ticker(state: AgentState):
    with mlflow.start_span(name="resolve_ticker"): # MLflow Tracing[cite: 1]
        ticker = resolve_ticker(state["company_input"])
        return {"ticker": ticker}

def node_fetch_news_and_price(state: AgentState):
    with mlflow.start_span(name="fetch_news_price"):
        news = fetch_news(state["company_input"])
        prices = fetch_stock_price(state["ticker"])
        return {"news_data": news, "stock_data": prices}

def node_analyze_sentiment(state: AgentState):
    with mlflow.start_span(name="sentiment_analysis"):
        structured_llm = llm.with_structured_output(MarketResearchOutput)
        prompt = f"""
        Analyze the following for {state['company_input']} ({state['ticker']}):
        News: {state['news_data']}
        Stock Data: {state['stock_data']}
        """
        report = structured_llm.invoke(prompt)
        return {"final_report": report.dict()}

# Build the Graph[cite: 1]
workflow = StateGraph(AgentState)
workflow.add_node("resolve", node_resolve_ticker)
workflow.add_node("data_fetch", node_fetch_news_and_price)
workflow.add_node("analyze", node_analyze_sentiment)

workflow.set_entry_point("resolve")
workflow.add_edge("resolve", "data_fetch")
workflow.add_edge("data_fetch", "analyze")
workflow.add_edge("analyze", END)

app = workflow.compile()

In [ ]:
# Step 1 & 7: Execution[cite: 1]
inputs = {"company_input": "NVIDIA"}
result = app.invoke(inputs)

import json
print(json.dumps(result["final_report"], indent=4))

In [ ]:
{
    "company_name": "NVIDIA Corporation",
    "stock_code": "NVDA",
    "current_price": 875.25,
    "price_change_7d": "+5.4%",
    "price_change_30d": "+12.1%",
    "fifty_two_week_high": 974.00,
    "fifty_two_week_low": 300.12,
    "news_summary": "NVIDIA chips see massive demand in data centers...",
    "sentiment": "Positive",
    "confidence_score": 0.95,
    "people_names": ["Jensen Huang"],
    "places_names": ["Santa Clara"],
    "other_companies_referred": ["AMD", "Intel"],
    "related_industries": ["Semiconductors", "AI", "Cloud"],
    "market_implications": "Dominance in AI hardware continues to drive revenue.",
    "investment_recommendation": "Buy",
    "recommendation_rationale": "Strong financial growth and undisputed market leadership."
}